<a href="https://colab.research.google.com/github/faisu6339-glitch/LLMs/blob/main/Bahdanau_%26_Luong_Attention_Mechanism.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Bahdanau Attention Mechanism Explained

The Bahdanau attention mechanism, also known as additive attention, was introduced in 2015 by Bahdanau et al. in their paper "Neural Machine Translation by Jointly Learning to Align and Translate." It was a significant breakthrough in sequence-to-sequence models, especially for tasks like machine translation, as it allowed the model to focus on relevant parts of the input sequence when generating each part of the output sequence.

Here's a detailed breakdown:

**1. The Problem Attention Solves (Context Vector Limitation):**
Traditional sequence-to-sequence models (encoder-decoder architectures) would encode the entire input sequence into a single fixed-size context vector. For long input sequences, this context vector became a bottleneck, struggling to retain all necessary information. The decoder would then generate the output solely based on this limited context vector, leading to information loss and difficulties in translating longer sentences.

**2. The Core Idea of Bahdanau Attention:**
Instead of compressing the entire input into a single context vector, Bahdanau attention allows the decoder to "look back" at all the encoder's hidden states at each step of decoding. It then calculates a weighted sum of these hidden states, where the weights determine how much attention (or importance) the decoder should place on each part of the input when generating the current output word.

**3. Key Components and Steps:**

*   **Encoder Hidden States (h_j):** The encoder processes the input sequence and generates a sequence of hidden states, one for each input token. Let's denote these as `h_1, h_2, ..., h_T_x`, where `T_x` is the length of the input sequence.

*   **Decoder Hidden State (s_i-1):** At each decoding step `i`, the decoder produces its own hidden state, `s_i`. For calculating attention at step `i`, we use the previous decoder hidden state, `s_i-1`.

*   **Alignment Score (e_ij):** This is the heart of the attention mechanism. For each decoder hidden state `s_i-1` and each encoder hidden state `h_j`, an alignment score `e_ij` is calculated. This score measures how well the `j`-th input hidden state `h_j` aligns with the current decoder state `s_i-1` (i.e., how relevant `h_j` is for predicting the next output word). In Bahdanau attention, this is typically computed using a feed-forward neural network:
    `e_ij = v_a^T * tanh(W_a * s_i-1 + U_a * h_j)`
    Where:
    *   `v_a`, `W_a`, `U_a` are learnable weight matrices and vectors.
    *   `tanh` is the activation function.
    *   This essentially combines `s_i-1` and `h_j` and passes them through a small neural network to get a scalar score.

*   **Attention Weights (alpha_ij):** The raw alignment scores `e_ij` are then normalized using a softmax function to produce attention weights `alpha_ij`. These weights sum to 1 across all encoder hidden states for a given decoder step `i`:
    `alpha_ij = exp(e_ij) / sum_k(exp(e_ik))`
    These `alpha_ij` values are the actual attention scores, indicating the probability or importance of `h_j` for the current decoding step.

*   **Context Vector (c_i):** Finally, a context vector `c_i` is computed as a weighted sum of all encoder hidden states, using the attention weights `alpha_ij`:
    `c_i = sum_j(alpha_ij * h_j)`
    This context vector `c_i` is a dynamically generated representation of the most relevant parts of the input sequence for generating the `i`-th output token.

*   **Concatenation and Decoder Input:** The context vector `c_i` is then concatenated with the decoder's previous hidden state `s_i-1` (or the previous output and previous hidden state, depending on the exact decoder architecture) and fed into the decoder to predict the next output word. This allows the decoder to make its prediction based on both its internal state and the specific relevant parts of the input.

**4. Key Characteristics of Bahdanau Attention:**

*   **Additive/Concatenative:** It combines the decoder's previous hidden state and the encoder's hidden states by concatenating them and passing them through a feed-forward network. This is why it's often called "additive attention."
*   **Pre-output Attention:** The attention mechanism calculates the context vector *before* the decoder produces its output for the current step.
*   **Soft Attention:** The attention weights are probabilities, allowing the model to smoothly attend to multiple input parts simultaneously, rather than focusing exclusively on one.
*   **Learnable:** The weights `v_a`, `W_a`, `U_a` are learned during the training process, allowing the model to adaptively determine what to focus on.

**5. Advantages:**

*   **Handles Long Sequences:** Overcomes the information bottleneck of fixed-size context vectors.
*   **Improved Performance:** Leads to significantly better performance in tasks like machine translation, especially for longer sentences.
*   **Interpretability:** The attention weights provide some degree of interpretability, showing which parts of the input the model focused on when generating each part of the output.

**6. Bahdanau vs. Luong Attention:**
It's important to note that Bahdanau attention is one of the earliest and most influential attention mechanisms. Another common type is Luong attention (multiplicative attention), which differs primarily in how the alignment scores are calculated (often using a dot product or general function) and where the context vector is used in the decoder (often after the decoder's hidden state has been computed for the current step).

In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

class BahdanauAttention(layers.Layer):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        self.W1 = layers.Dense(units)
        self.W2 = layers.Dense(units)
        self.V = layers.Dense(1)

    def call(self, query, values):
        # query hidden state shape == (batch_size, hidden_size)
        # values shape == (batch_size, max_len, hidden_size)

        # hidden_with_time_axis shape == (batch_size, 1, hidden_size)
        # we are doing this to broadcast addition along the time axis to the 'values'
        hidden_with_time_axis = tf.expand_dims(query, 1)

        # score shape == (batch_size, max_len, 1)
        # we apply self.V to the tanh output of the concatenated values and query
        score = self.V(tf.nn.tanh(self.W1(values) + self.W2(hidden_with_time_axis)))

        # attention_weights shape == (batch_size, max_len, 1)
        attention_weights = tf.nn.softmax(score, axis=1)

        # context_vector shape == (batch_size, hidden_size)
        context_vector = attention_weights * values
        context_vector = tf.reduce_sum(context_vector, axis=1)

        return context_vector, attention_weights

### Explanation of the `BahdanauAttention` Class:

*   **`__init__(self, units)`**:
    *   Initializes three `layers.Dense` (fully connected) layers: `W1`, `W2`, and `V`.
    *   `units` defines the dimensionality of the intermediate representation. The choice of `units` is a hyperparameter.
    *   `W1` is applied to the `values` (encoder hidden states).
    *   `W2` is applied to the `query` (decoder hidden state).
    *   `V` is applied to the combined result of `W1` and `W2` to produce a scalar score for each attention alignment.

*   **`call(self, query, values)`**: This method defines the forward pass of the attention mechanism.
    *   **`query`**: Represents the current hidden state of the decoder (`s_i-1` in the explanation). Its shape is `(batch_size, hidden_size)`.
    *   **`values`**: Represents all the hidden states from the encoder (`h_j` in the explanation). Its shape is `(batch_size, max_len, hidden_size)`.
    *   **`hidden_with_time_axis = tf.expand_dims(query, 1)`**: The decoder's `query` state is expanded to have a time dimension, so it can be added to the `values` (encoder states) which have a time dimension. This allows for broadcasting the `query` across all time steps of the `values`.
    *   **`score = self.V(...)`**: This is where the alignment score (`e_ij`) is calculated.
        *   `self.W1(values)`: Transforms the encoder hidden states.
        *   `self.W2(hidden_with_time_axis)`: Transforms the decoder hidden state.
        *   These transformed states are added together (broadcast across `max_len`).
        *   `tf.nn.tanh(...)`: An activation function (tanh) is applied to the sum.
        *   `self.V(...)`: The result is then passed through the `V` dense layer, which outputs a scalar score for each encoder hidden state, indicating its relevance to the current decoder state. The `score` shape is `(batch_size, max_len, 1)`.
    *   **`attention_weights = tf.nn.softmax(score, axis=1)`**: The scores are normalized using `softmax` along the `max_len` axis to get the attention weights (`alpha_ij`). These weights sum to 1 for each sequence in the batch. The `attention_weights` shape is `(batch_size, max_len, 1)`.
    *   **`context_vector = attention_weights * values`**: Each encoder hidden state in `values` is multiplied by its corresponding attention weight. This weighs the importance of each encoder state. The shape remains `(batch_size, max_len, hidden_size)`.
    *   **`context_vector = tf.reduce_sum(context_vector, axis=1)`**: The weighted encoder states are summed along the `max_len` (time) axis to produce the final `context_vector` (`c_i`). This vector is a summary of the most relevant parts of the input sequence for the current decoding step. Its shape is `(batch_size, hidden_size)`.
    *   **`return context_vector, attention_weights`**: The method returns both the computed `context_vector` and the `attention_weights` (which can be useful for visualization or debugging).

## Luong Attention Mechanism Explained

Luong attention, introduced by Luong et al. in 2015 in their paper "Effective Approaches to Attention-based Neural Machine Translation," is another prominent type of attention mechanism used in sequence-to-sequence models. While it shares the same goal as Bahdanau attention (to weigh the importance of encoder states for decoder output), it differs in its formulation, particularly in the calculation of alignment scores and how the context vector is used.

**1. Key Components and Steps (similar to Bahdanau but with key differences):**

*   **Encoder Hidden States (h_j):** As before, the encoder produces a sequence of hidden states `h_1, h_2, ..., h_T_x`.

*   **Decoder Hidden State (s_i):** At each decoding step `i`, the decoder produces its current hidden state, `s_i`. **A key difference from Bahdanau attention is that Luong attention typically uses the *current* decoder hidden state `s_i` (instead of `s_i-1`) to compute the attention weights.**

*   **Alignment Score (e_ij):** This is where Luong attention offers several different "scoring functions." The score `e_ij` measures the similarity between the decoder's current hidden state `s_i` and each encoder hidden state `h_j`.
    
    Luong proposed three main scoring functions:
    *   **Dot Product:** `e_ij = s_i^T * h_j` (simple dot product, works best when `s_i` and `h_j` have the same dimensionality)
    *   **General:** `e_ij = s_i^T * W_a * h_j` (introduces a learnable weight matrix `W_a`)
    *   **Concat:** `e_ij = v_a^T * tanh(W_a * [s_i; h_j])` (similar to Bahdanau's additive style, but typically uses `s_i`)
    
    The most common and often preferred are 'dot' and 'general'.

*   **Attention Weights (alpha_ij):** The alignment scores `e_ij` are normalized using a softmax function to get the attention weights `alpha_ij`:
    `alpha_ij = exp(e_ij) / sum_k(exp(e_ik))`

*   **Context Vector (c_i):** The context vector `c_i` is computed as a weighted sum of the encoder hidden states, identical to Bahdanau attention:
    `c_i = sum_j(alpha_ij * h_j)`

*   **Concatenation and Decoder Output:** Another key difference is how the context vector is used. In Luong attention, the context vector `c_i` is often *concatenated* with the **current** decoder hidden state `s_i` to form a new, attention-aware hidden state, `s'_i`. This `s'_i` is then used to predict the output word for the current step `i`.
    `s'_i = tanh(W_c * [c_i; s_i])`
    Where `W_c` is a learnable weight matrix.

**2. Key Characteristics of Luong Attention:**

*   **Multiplicative/Dot-Product (often):** The 'dot' and 'general' scoring functions involve multiplications of states, giving it a "multiplicative" flavor, as opposed to Bahdanau's "additive" nature.
*   **Post-output Attention:** The attention mechanism typically calculates the context vector and combines it with the decoder's *current* hidden state *after* the decoder has processed its input for the current step and produced a hidden state.
*   **"Global" vs. "Local" Attention:** Luong also introduced the concepts of global and local attention. Global attention (what we've described) attends to all encoder hidden states. Local attention focuses only on a subset of encoder hidden states around a predicted alignment point, which can be more computationally efficient for very long sequences.

**3. Main Differences from Bahdanau Attention:**

*   **Decoder State Usage:** Luong uses the *current* decoder hidden state (`s_i`) for alignment score calculation, whereas Bahdanau uses the *previous* decoder hidden state (`s_i-1`).
*   **Alignment Score Calculation:** Luong attention often employs simpler scoring functions (dot, general) directly comparing `s_i` and `h_j`, while Bahdanau uses an additive, feed-forward network approach with `tanh`.
*   **Context Vector Integration:** In Luong, the context vector `c_i` is typically concatenated with the *current* decoder hidden state `s_i` to produce a 'filtered' or 'attended' decoder state `s'_i` from which the output is predicted. In Bahdanau, `c_i` is directly fed into the decoder for the next step.
*   **Name:** Bahdanau is often called "additive attention," and Luong (especially with dot/general) is called "multiplicative attention."

Let's implement the 'general' variant of Luong attention, as it's quite common.

In [3]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

class LuongAttention(layers.Layer):
    def __init__(self, units, attention_type='general'):
        super(LuongAttention, self).__init__()
        self.units = units
        self.attention_type = attention_type

        if self.attention_type == 'general':
            # W_a in e_ij = s_i^T * W_a * h_j
            self.W_attn = layers.Dense(self.units, use_bias=False)
        elif self.attention_type == 'concat':
            # Similar to Bahdanau, but using current decoder state (s_i)
            self.W1 = layers.Dense(units)
            self.W2 = layers.Dense(units)
            self.V = layers.Dense(1)
        # For 'dot' attention, no additional weights are needed here

    def call(self, query, values):
        # query hidden state shape == (batch_size, hidden_size) - this is s_i
        # values shape == (batch_size, max_len, hidden_size) - these are h_j

        if self.attention_type == 'dot':
            # score shape == (batch_size, max_len)
            # s_i . h_j (transpose h_j)
            query = tf.expand_dims(query, 1) # (batch_size, 1, hidden_size)
            score = tf.matmul(query, values, transpose_b=True)
            score = tf.squeeze(score, axis=1) # (batch_size, max_len)

        elif self.attention_type == 'general':
            # score shape == (batch_size, max_len)
            # s_i . (W_a * h_j)
            # W_a * h_j needs to be applied to each h_j first
            transformed_values = self.W_attn(values) # (batch_size, max_len, units)
            query = tf.expand_dims(query, 1) # (batch_size, 1, hidden_size)
            score = tf.matmul(query, transformed_values, transpose_b=True)
            score = tf.squeeze(score, axis=1) # (batch_size, max_len)

        elif self.attention_type == 'concat':
            # query hidden state shape == (batch_size, 1, hidden_size)
            hidden_with_time_axis = tf.expand_dims(query, 1)

            # score shape == (batch_size, max_len, 1)
            score = self.V(tf.nn.tanh(self.W1(values) + self.W2(hidden_with_time_axis)))
            score = tf.squeeze(score, axis=-1) # (batch_size, max_len)

        else:
            raise ValueError("Invalid attention_type. Must be 'dot', 'general', or 'concat'.")

        # attention_weights shape == (batch_size, max_len)
        attention_weights = tf.nn.softmax(score, axis=1)

        # context_vector shape == (batch_size, max_len, hidden_size)
        # after expand_dims: (batch_size, max_len, 1) * (batch_size, max_len, hidden_size)
        context_vector = tf.expand_dims(attention_weights, axis=-1) * values

        # context_vector shape == (batch_size, hidden_size)
        context_vector = tf.reduce_sum(context_vector, axis=1)

        return context_vector, attention_weights

### Explanation of the `LuongAttention` Class:

*   **`__init__(self, units, attention_type='general')`**:
    *   Takes `units` (for dimensionality, especially important for 'general' and 'concat' types) and `attention_type` (defaulting to 'general') as arguments.
    *   If `attention_type` is 'general', it initializes a `W_attn` `Dense` layer which corresponds to the `W_a` matrix in the `s_i^T * W_a * h_j` scoring function. `use_bias=False` is common for this weight matrix.
    *   If `attention_type` is 'concat', it initializes layers similar to Bahdanau, but these will be used with the *current* decoder state `s_i`.

*   **`call(self, query, values)`**: This method defines the forward pass.
    *   **`query`**: Represents the *current* hidden state of the decoder (`s_i`). Shape: `(batch_size, hidden_size)`.
    *   **`values`**: Represents all the hidden states from the encoder (`h_j`). Shape: `(batch_size, max_len, hidden_size)`.

    *   **Scoring Function Logic (conditional based on `attention_type`):**
        *   **`'dot'`:**
            *   The `query` is expanded to `(batch_size, 1, hidden_size)`.
            *   `tf.matmul(query, values, transpose_b=True)` performs the dot product `query . values^T`. This effectively computes `s_i . h_j` for all `h_j` in a vectorized manner. The output shape is `(batch_size, 1, max_len)`.
            *   `tf.squeeze(score, axis=1)` removes the redundant dimension, resulting in `(batch_size, max_len)`.
        *   **`'general'`:**
            *   `transformed_values = self.W_attn(values)`: Each encoder hidden state `h_j` is first transformed by `W_a`. This results in `W_a * h_j` for all `j`. The shape becomes `(batch_size, max_len, units)`.
            *   The `query` is expanded, and `tf.matmul` is used to compute `query . (W_a * values)^T`. Resulting shape `(batch_size, max_len)`.
        *   **`'concat'`:**
            *   This path is very similar to the Bahdanau `call` method, but remember `query` here is `s_i` (current decoder state) not `s_i-1`.
            *   The `score` is initially `(batch_size, max_len, 1)` and then `tf.squeeze` is used to get `(batch_size, max_len)`.

    *   **`attention_weights = tf.nn.softmax(score, axis=1)`**: The calculated scores are normalized using `softmax` along the `max_len` axis to get the attention weights. Shape: `(batch_size, max_len)`.

    *   **`context_vector = tf.expand_dims(attention_weights, axis=-1) * values`**: The `attention_weights` are expanded to `(batch_size, max_len, 1)` to allow element-wise multiplication with `values` (encoder states). This calculates the weighted encoder states. Shape: `(batch_size, max_len, hidden_size)`.

    *   **`context_vector = tf.reduce_sum(context_vector, axis=1)`**: The weighted encoder states are summed along the `max_len` (time) axis to produce the final `context_vector`. Shape: `(batch_size, hidden_size)`.

    *   **`return context_vector, attention_weights`**: Returns the `context_vector` and the `attention_weights`.

In [4]:
print('Luong attention layer implementation added to your notebook.')

Luong attention layer implementation added to your notebook.


In [2]:
print('Bahdanau attention layer implementation added to your notebook.')

Bahdanau attention layer implementation added to your notebook.
